In [1]:
import binpan

In [2]:
!python -V

Python 3.13.12


In [3]:
binpan.__version__

'0.8.14'

# Tagging and Backtesting

BinPan includes tools for labeling trading signals and backtesting simple strategies. This notebook demonstrates:

1. **EMA Cross Strategy**: generating buy/sell signals from moving average crossovers
2. **Stop Loss & Target**: applying custom stop-loss and take-profit levels
3. **Entry Filters**: filtering entries based on additional conditions
4. **Backtesting**: simulating trades with fees and evaluating performance (ROI, profit/hour)

We start by creating a Symbol, adding two EMAs (fast=7, slow=50), and computing their crossover signals.

In [4]:
my_symbol = 'LTCUSDT'

sym = binpan.Symbol(symbol=my_symbol, tick_interval='1h', time_zone='Europe/Madrid', limit=1000)
sym.ema(window=7, color='darkgrey')
sym.ema(window=50, color='black')
sym.cross(fast='EMA_7', slow='EMA_50').dropna()

2026-03-05	 20:54:12     INFO get_candles_by_time_stamps -> symbol=LTCUSDT tick_interval=1h start=2026-01-23 03:00:00 end=2026-03-05 19:59:59 limit=1000
2026-03-05 20:54:12     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-05 20:54:13     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90
2026-03-05	 20:54:18     INFO New column: EMA_7
2026-03-05	 20:54:19     INFO New column: EMA_50
2026-03-05	 20:54:19     INFO New column: Cross_EMA_50_EMA_7


LTCUSDT 1h Europe/Madrid
2026-01-23 04:00:00+01:00    1.0
2026-01-23 07:00:00+01:00   -1.0
2026-01-24 14:00:00+01:00    1.0
2026-01-24 17:00:00+01:00   -1.0
2026-01-25 08:00:00+01:00    1.0
                            ... 
2026-03-01 17:00:00+01:00   -1.0
2026-03-02 16:00:00+01:00    1.0
2026-03-03 09:00:00+01:00   -1.0
2026-03-03 18:00:00+01:00    1.0
2026-03-05 18:00:00+01:00   -1.0
Name: Cross_EMA_50_EMA_7, Length: 44, dtype: float64

In [5]:
sym.plot(actions_col='Cross_EMA_50_EMA_7')

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

# Visualizing the Strategy

The `plot()` method with `actions_col` overlays buy (+1) and sell (-1) markers on the candlestick chart, making it easy to visually inspect the strategy's entry and exit points.

In [6]:
# Simple backtesting without stop loss or target (uncomment to try):
# sym.backtesting(actions_col='Cross_EMA_50_EMA_7', fee=0.001, base=0, quote=1000)

In [7]:
# Plot with custom marker labels (uncomment to try):
# sym.plot(actions_col='Cross_EMA_50_EMA_7', priced_actions_col='Close', marker_labels={1:'buy', -1:'sell'})

# Stop Loss & Target

Custom stop-loss and target series are created and inserted as indicator columns. These will be used by the backtesting engine to simulate realistic trade exits:
- **Target**: 1% above the Open price (`Open * 1.01`)
- **Trailing Stop Loss**: minimum Low over the last 5 candles

The `insert_indicator()` method adds any custom Series to the Symbol's DataFrame with configurable plot appearance.

In [8]:
target = sym.df['Open'] * 1.01
trailing_stop_loss = sym.df['Low'].rolling(5).min()

In [9]:
sym.insert_indicator(source_data=target, plotting_rows=[1], name='target', color='magenta')
sym.insert_indicator(source_data=trailing_stop_loss, plotting_rows=[1], name='stop_loss', color='blue')

2026-03-05	 20:54:21     INFO New column: target
2026-03-05	 20:54:21     INFO New column: stop_loss


,Open time,Open,High,Low,Close,Volume,Close time,Quote volume,Trades,Taker buy base volume,Taker buy quote volume,Ignore,Open timestamp,Close timestamp,EMA_7,EMA_50,Cross_EMA_50_EMA_7,target,stop_loss
LTCUSDT 1h Europe/Madrid,,,,,,,,,,,,,,,,,,,
2026-01-23 03:00:00+01:00,2026-01-23 03:00:00+01:00,69.48,69.60,68.90,69.04,15017.984,2026-01-23 03:59:59.999000+01:00,1.040553e+06,9055,7301.251,5.059577e+05,0,1769133600000,1769137199999,69.040000,69.040000,NaN,70.1748,NaN
2026-01-23 04:00:00+01:00,2026-01-23 04:00:00+01:00,69.03,69.60,69.00,69.34,19052.285,2026-01-23 04:59:59.999000+01:00,1.321729e+06,7952,9998.001,6.937149e+05,0,1769137200000,1769140799999,69.115000,69.051765,1.0,69.7203,NaN
2026-01-23 05:00:00+01:00,2026-01-23 05:00:00+01:00,69.34,69.49,69.18,69.35,11993.270,2026-01-23 05:59:59.999000+01:00,8.313356e+05,6726,5400.934,3.744039e+05,0,1769140800000,1769144399999,69.173750,69.063460,NaN,70.0334,NaN
2026-01-23 06:00:00+01:00,2026-01-23 06:00:00+01:00,69.34,69.34,68.86,69.01,30382.117,2026-01-23 06:59:59.999000+01:00,2.096493e+06,7779,17593.634,1.213710e+06,0,1769144400000,1769147999999,69.132813,69.061364,NaN,70.0334,NaN
2026-01-23 07:00:00+01:00,2026-01-23 07:00:00+01:00,69.01,69.16,68.68,68.74,9096.453,2026-01-23 07:59:59.999000+01:00,6.271065e+05,7008,4812.327,3.317910e+05,0,1769148000000,1769151599999,69.034609,69.048761,-1.0,69.7001,68.68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-05 15:00:00+01:00,2026-03-05 15:00:00+01:00,55.97,56.61,55.92,56.49,27333.194,2026-03-05 15:59:59.999000+01:00,1.538207e+06,18806,13140.944,7.396535e+05,0,1772719200000,1772722799999,56.302972,55.872344,NaN,56.5297,55.48
2026-03-05 16:00:00+01:00,2026-03-05 16:00:00+01:00,56.49,56.59,55.67,55.89,29391.688,2026-03-05 16:59:59.999000+01:00,1.647408e+06,18894,11555.070,6.475444e+05,0,1772722800000,1772726399999,56.199729,55.873037,NaN,57.0549,55.48
2026-03-05 17:00:00+01:00,2026-03-05 17:00:00+01:00,55.90,55.91,55.05,55.48,27410.527,2026-03-05 17:59:59.999000+01:00,1.520130e+06,18208,11248.014,6.239278e+05,0,1772726400000,1772729999999,56.019797,55.857623,NaN,56.4590,55.05


# Entry Filter

The `tag()` method creates boolean labels based on comparisons between columns. Here we require that the Low price must be above EMA_7 for a buy signal to be valid. This filters out entries in weak momentum conditions.

In [10]:
sym.tag(column='Low', reference='EMA_7', relation='gt')

2026-03-05	 20:54:22     INFO New column: Tag_Low_gt_EMA_7


LTCUSDT 1h Europe/Madrid
2026-01-23 03:00:00+01:00    0
2026-01-23 04:00:00+01:00    0
2026-01-23 05:00:00+01:00    1
2026-01-23 06:00:00+01:00    0
2026-01-23 07:00:00+01:00    0
                            ..
2026-03-05 15:00:00+01:00    0
2026-03-05 16:00:00+01:00    0
2026-03-05 17:00:00+01:00    0
2026-03-05 18:00:00+01:00    0
2026-03-05 19:00:00+01:00    0
Name: Tag_Low_gt_EMA_7, Length: 1001, dtype: int64

# Backtesting

The `backtesting()` method simulates a long-only strategy with:
- **Initial portfolio**: 0 base asset, 1000 quote (USDT)
- **Fee**: 0.1% per trade
- **Stop loss/target**: applied per-trade with fixed values at entry time
- **Entry filter**: only buy when `Tag_Low_gt_EMA_7` is active

The result includes wallet balances, evaluated portfolio value, and executed trade details.

In [11]:
sym.backtesting(actions_col='Cross_EMA_50_EMA_7', 
                fee=0.001, 
                base=0, 
                quote=1000, 
                target_column=target, 
                stop_loss_column=trailing_stop_loss,
                fixed_stop_loss=True,
                evaluating_quote='USDT',
                entry_filter_column='Tag_Low_gt_EMA_7')

2026-03-05	 20:54:22     INFO New column: Wallet_baseNone
2026-03-05	 20:54:22     INFO New column: Wallet_quoteNone
2026-03-05	 20:54:22     INFO New column: Evaluated_LTCUSDT_in_USDT
2026-03-05	 20:54:22     INFO New column: Resulting_actions_Cross_EMA_50_EMA_7
2026-03-05	 20:54:22     INFO New column: Executed_prices_Cross_EMA_50_EMA_7


,Wallet_baseNone,Wallet_quoteNone,Evaluated_LTCUSDT_in_USDT,Resulting_actions_Cross_EMA_50_EMA_7,Executed_prices_Cross_EMA_50_EMA_7
LTCUSDT 1h Europe/Madrid,,,,,
2026-01-23 03:00:00+01:00,0.000000,1000.000000,1000.000000,NaN,NaN
2026-01-23 04:00:00+01:00,0.000000,1000.000000,1000.000000,NaN,NaN
2026-01-23 05:00:00+01:00,14.407269,0.000000,999.144073,1.0,69.34
2026-01-23 06:00:00+01:00,14.407269,0.000000,994.245601,NaN,NaN
2026-01-23 07:00:00+01:00,14.407269,0.000000,990.355639,NaN,NaN
...,...,...,...,...,...
2026-03-05 15:00:00+01:00,0.000000,1000.485452,1000.485452,NaN,NaN
2026-03-05 16:00:00+01:00,0.000000,1000.485452,1000.485452,NaN,NaN
2026-03-05 17:00:00+01:00,0.000000,1000.485452,1000.485452,NaN,NaN


## Understanding Action Labels

The backtesting engine produces three types of actions:
- **+1 (buy)**: a regular buy entry
- **-1 (sell)**: a regular sell exit
- **+2 (FAST)**: both entry and exit occurred within the same candle (stop loss or target was hit immediately)

These are plotted with custom marker labels for easy visual identification.

In [12]:
sym.df['Resulting_actions_Cross_EMA_50_EMA_7'].value_counts()

Resulting_actions_Cross_EMA_50_EMA_7
 1.0    13
-1.0    13
 2.0     2
Name: count, dtype: int64

In [13]:
my_marker_labels={-1:'sell', 1:'buy', 2: 'FAST'}

In [14]:
sym.plot(actions_col='Resulting_actions_Cross_EMA_50_EMA_7', priced_actions_col='Executed_prices_Cross_EMA_50_EMA_7', marker_labels=my_marker_labels)

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

# Results

The `roi()` method calculates the total return on investment, and `profit_hour()` returns the raw profit ratio per hour (not annualized). These metrics help evaluate whether the strategy is worth pursuing.

In [15]:
# find evaluated column for calculation of returns
eval_col = [i for i in list(sym.df.columns) if i.startswith('Eval')][0]
eval_col

'Evaluated_LTCUSDT_in_USDT'

In [16]:
sym.roi(column=eval_col)

np.float64(0.04854519079710826)

In [17]:
sym.profit_hour(eval_col)

Total profit for Evaluated_LTCUSDT_in_USDT: 0.4854519079710826 with ratio 0.000484966941164631 per hour.


np.float64(0.000484966941164631)